# 第20講：数値変数・カテゴリ変数の変換

現実のデータ表には**文字データ**（カテゴリ変数）と**単位の異なる数値データ**（数値変数）が混在している。
第21講以降で学ぶ**主成分分析・線形回帰・ニューラルネットワーク**などの機械学習モデルにデータを正しく学習させるため、このノートブックでは**4つの実践タスク**を順番に進め、各タスクの直後に**実習①〜実習④**を行う。

1. **タスク①＋実習①**：2値カテゴリ・順序カテゴリの変換と二値化（高校データ `mhigh.csv`）
2. **タスク②＋実習②**：3種類以上の名義カテゴリとダミー変数（ペンギンデータ `penguin.csv`・アヤメデータ `iris.csv`）
3. **タスク③＋実習③**：数値変数の標準化と正規化（ペンギンデータ `penguin.csv`）
4. **タスク④＋実習④**：歪んだ数値の対数変換と機械学習での効果検証（賃金データ `CPS1988.csv`・ペンギンAI分類）

## 0. 準備

最初に1回だけ実行する。グラフの軸に日本語を出すため、Noto Sans JP を登録する。

In [ ]:
import piplite
import pyodide
await piplite.install(['seaborn'])

import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from pyodide.http import open_url
%matplotlib inline

font_file = "NotoSansJP-Bold.otf"
with open(font_file, "wb") as f:
    f.write(open_url("https://raw.githubusercontent.com/ChungWookyung/kg-jupyterlite-data-analysis/main/content/DATA/NotoSansJP-Bold.otf").getvalue().encode('latin1'))
fm.fontManager.addfont(font_file)
mpl.rcParams['font.family'] = fm.FontProperties(fname=font_file).get_name()
mpl.rcParams['axes.unicode_minus'] = False

---
## 1. タスク①：カテゴリ変数の数値化と二値化（高校データ `mhigh.csv`）

まずは高校677人の擬似データ（`mhigh.csv`）を読み込み、列名を英語に揃えて中身を確認する。

In [ ]:
url_mh = "https://raw.githubusercontent.com/ChungWookyung/kg-jupyterlite-data-analysis/main/content/DATA/mhigh.csv"
mhigh = pd.read_csv(open_url(url_mh))
mhigh.columns = ["name", "katakana", "sex", "grade",
                 "japanese", "math", "total", "height", "weight", "class"]
mhigh.head(3)

### 1-1. 不要な列を `.drop()` で取り除く

機械学習では、個人名（`name`, `katakana`）のような予測に無関係な列や、合計点（`total`）のように答えそのものが含まれる列は `drop(..., axis=1)` で除外する。

In [ ]:
# 氏名・カタカナ・成績合計を取り除いた表を確認する
df1 = mhigh.drop(["name", "katakana", "total"], axis=1)
df1.head(3)

### 1-2. 2値カテゴリを 0 と 1 に変換する（`.replace`）

選択肢が2種類だけ（`"女"` / `"男"`）のカテゴリ変数は、`.replace({"女": 1, "男": 0})` で1列のまま `1` と `0` に変換する。
0と1に変換すると、合計 `.sum()` が「1の人数」、平均 `.mean()` が「1の割合」になる。

In [ ]:
# 性別（"女" / "男"）を 1 と 0 に変換する
mhigh["sex_num"] = mhigh["sex"].replace({"女": 1, "男": 0})
print(mhigh["sex_num"].value_counts())
print("女子の人数:", mhigh["sex_num"].sum(), "人")
print("女子の割合:", mhigh["sex_num"].mean().round(4))

### 1-3. 順序カテゴリを順序数に変換する

`class`（`"A"`〜`"H"`）のように**ランクや順番に意味があるカテゴリ変数（順序カテゴリ）**は、大小関係を保ったまま `8〜1` の数値に変換する。

In [ ]:
# クラスごとの数学平均点を確認する
print(mhigh.groupby("class")["math"].mean().round(1))

# A=8, B=7, ..., H=1 の順序数に変換する
class_map = {"A": 8, "B": 7, "C": 6, "D": 5, "E": 4, "F": 3, "G": 2, "H": 1}
mhigh["class_num"] = mhigh["class"].replace(class_map)
mhigh[["name", "class", "class_num", "math"]].head(5)

### 1-4. 数値変数から 0・1 のラベルを作る（二値化）

「数学が60点以上なら合格（`1`）、60点未満なら不合格（`0`）」のように、条件式に `.astype(int)` を付けると分類問題の目的変数（正解ラベル）を作ることができる。

In [ ]:
# 数学が60点以上なら 1、未満なら 0 にする（二値化）
mhigh["math_pass"] = (mhigh["math"] >= 60).astype(int)
print("数学60点以上の合格者数:", mhigh["math_pass"].sum(), "人")
print("数学60点以上の合格率:", mhigh["math_pass"].mean().round(4))

### 1-5. カテゴリ変数の数値化で予測精度がどう改善するか確認する

数値だけ（国語・身長・体重・学年）で数学の点数を予測した場合と、数値化した性別（`sex_num`）とクラス（`class_num`）を追加した場合で、テストデータの予測誤差（MAE）を比較する。

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

train_mh = mhigh.sample(frac=0.8, random_state=42)
test_mh = mhigh.drop(train_mh.index)

cols_num = ["japanese", "height", "weight", "grade"]
reg1 = LinearRegression().fit(train_mh[cols_num], train_mh["math"])
mae1 = mean_absolute_error(test_mh["math"], reg1.predict(test_mh[cols_num]))

cols_all = ["japanese", "height", "weight", "grade", "sex_num", "class_num"]
reg2 = LinearRegression().fit(train_mh[cols_all], train_mh["math"])
mae2 = mean_absolute_error(test_mh["math"], reg2.predict(test_mh[cols_all]))

print("数値のみの誤差 MAE:", round(mae1, 2), "点")
print("性別・クラス追加後の誤差 MAE:", round(mae2, 2), "点")

### 実習①：高校データのカテゴリ変換と二値化

次の4つを下のコードセルで実行しなさい。
1. `mhigh.csv` の `sex` 列を `.replace({"女": 1, "男": 0})` で数値化した列 **`sex_num`** を作り、女子の人数（`.sum()`）と割合（`.mean().round(4)`）を表示する
2. クラス `A〜H` を `8〜1` に変換した列 **`class_num`** を作る
3. 国語（`japanese`）が **60点以上なら `1`、未満なら `0`** となる二値化列 **`jp_pass`** を `(mhigh["japanese"] >= 60).astype(int)` で作る
4. 国語60点以上の人数（`mhigh["jp_pass"].sum()`）と割合（`mhigh["jp_pass"].mean().round(4)`）を表示する

In [ ]:
# 実習①：ここにコードを書いて実行する
mhigh["sex_num"] = mhigh["sex"].replace({"女": 1, "男": 0})
print("女子の人数:", mhigh["sex_num"].sum(), "割合:", mhigh["sex_num"].mean().round(4))

class_map = {"A": 8, "B": 7, "C": 6, "D": 5, "E": 4, "F": 3, "G": 2, "H": 1}
mhigh["class_num"] = mhigh["class"].replace(class_map)

mhigh["jp_pass"] = (mhigh["japanese"] >= 60).astype(int)
print("国語60点以上の人数:", mhigh["jp_pass"].sum(), "割合:", mhigh["jp_pass"].mean().round(4))

---
## 2. タスク②：名義カテゴリとダミー変数化（ペンギンデータ `penguin.csv`・アヤメデータ `iris.csv`）

生息する島（`Biscoe`, `Dream`, `Torgersen`）や品種（`Adelie`, `Chinstrap`, `Gentoo`）のように、**順序に意味がない3種類以上のカテゴリ（名義カテゴリ）**に `1, 2, 3` を割り当てると、AIが「3は1の3倍である」と誤解してしまう。
そこで、`pd.get_dummies(..., dtype=int)` を使って、カテゴリごとに `1` と `0` の列を作る**ダミー変数化（One-Hot Encoding）**を行う。

In [ ]:
url_pen = "https://raw.githubusercontent.com/ChungWookyung/kg-jupyterlite-data-analysis/main/content/DATA/penguin.csv"
penguin = pd.read_csv(open_url(url_pen))
penguin.head(3)

### 2-1. 不要な `year` 列の削除・`sex` の0/1変換・`island` と `species` のダミー変数化

In [ ]:
# 1. 分類に不要な観測年（year）列を削除する
penguin = penguin.drop("year", axis=1)

# 2. 2種類の性別（sex）は 1 と 0 に変換する
penguin["sex"] = penguin["sex"].replace({"female": 1, "male": 0})

# 3. 3種類ある island と species をダミー変数（0と1の列）に展開する
penguin_dum = pd.get_dummies(penguin, columns=["island", "species"], dtype=int)
penguin_dum.head(3)

In [ ]:
# ダミー変数の列ごとに合計（羽数）と平均（構成比）を確認する
print("各島の羽数:
", penguin_dum[["island_Biscoe", "island_Dream", "island_Torgersen"]].sum())
print("
各品種の割合:
", penguin_dum[["species_Adelie", "species_Chinstrap", "species_Gentoo"]].mean().round(3))

### 2-2. アヤメデータ（`iris.csv`）の品種をダミー変数化する

第26講〜第28講のプロジェクト実習で使う `iris.csv`（150本のアヤメのデータ）でも、品種 `variety`（`Setosa`, `Versicolor`, `Virginica`）を `pd.get_dummies()` でダミー変数に変換する。

In [ ]:
url_iris = "https://raw.githubusercontent.com/ChungWookyung/kg-jupyterlite-data-analysis/main/content/DATA/iris.csv"
iris = pd.read_csv(open_url(url_iris))

iris_dum = pd.get_dummies(iris, columns=["variety"], dtype=int)
iris_dum.head(3)

### 実習②：ダミー変数（One-Hot Encoding）の作成

次の3つを下のコードセルで実行しなさい。
1. `penguin_dum` のすべての列名（`penguin_dum.columns.tolist()`）を表示し、全部で何列になったか確認する
2. `penguin_dum` の3つの島ダミー列（`island_Biscoe`, `island_Dream`, `island_Torgersen`）の合計（`.sum()`）を表示する
3. `iris.csv` をダミー変数化した `iris_dum` について、3つの品種ダミー列（`variety_Setosa`, `variety_Versicolor`, `variety_Virginica`）の合計（`.sum()`）と先頭5行（`.head()`）を表示する

In [ ]:
# 実習②：ここにコードを書いて実行する
print("penguin_dumの列数:", len(penguin_dum.columns), "列名:", penguin_dum.columns.tolist())
print(penguin_dum[["island_Biscoe", "island_Dream", "island_Torgersen"]].sum())

print(iris_dum[["variety_Setosa", "variety_Versicolor", "variety_Virginica"]].sum())
iris_dum.head()

---
## 3. タスク③：数値変数のスケーリング（標準化と正規化）

ペンギンデータの4つの身体測定値（`bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g`）は、ミリメートル（`mm`）とグラム（`g`）で単位が異なり、数値の大きさに約250倍の差がある。

In [ ]:
num_cols = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
penguin[num_cols].describe().loc[["mean", "std", "min", "max"]].round(2)

### 3-1. 標準化（Standardization）：平均を `0`、標準偏差を `1` に揃える

$$z = \frac{x - \text{平均}}{\text{標準偏差}}$$

Pandasでは `(df[num_cols] - df[num_cols].mean()) / df[num_cols].std()` で一括変換できる。

In [ ]:
# 4つの数値列をまとめて標準化（平均0・標準偏差1）する
penguin_std = penguin.copy()
penguin_std[num_cols] = (penguin[num_cols] - penguin[num_cols].mean()) / penguin[num_cols].std()

print("標準化後の平均:
", penguin_std[num_cols].mean().round(2))
print("
標準化後の標準偏差:
", penguin_std[num_cols].std().round(2))

### 3-2. 正規化（Min-Max Normalization）：最小値を `0`、最大値を `1` に揃える

$$x_{\text{norm}} = \frac{x - \text{最小値}}{\text{最大値} - \text{最小値}}$$

Pandasでは `(df[num_cols] - df[num_cols].min()) / (df[num_cols].max() - df[num_cols].min())` で一括変換できる。

In [ ]:
# 4つの数値列をまとめて正規化（最小値0・最大値1）する
penguin_norm = penguin.copy()
penguin_norm[num_cols] = (penguin[num_cols] - penguin[num_cols].min()) / (penguin[num_cols].max() - penguin[num_cols].min())

penguin_norm[num_cols].describe().loc[["min", "max", "mean"]].round(2)

### 3-3. 変換前・標準化・正規化の散布図を並べて比較する

点の並び方（3品種の位置関係や相関）は一切変わらず、**縦軸と横軸の目盛り（スケール）だけ**が揃っていることを確認しよう。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
for ax, df_cur, title in zip(
    axes,
    [penguin, penguin_std, penguin_norm],
    ["① 変換前 (mm と g)", "② 標準化 (平均0, 標準偏差1)", "③ 正規化 (最小0, 最大1)"]
):
    sns.scatterplot(data=df_cur, x="bill_length_mm", y="body_mass_g", hue="species", ax=ax)
    ax.set_title(title)
plt.tight_layout()
plt.show()

### 実習③：数値変数の標準化と正規化

次の3つを下のコードセルで実行しなさい。
1. `penguin_std[num_cols]` の平均（`.mean().round(2)`）と標準偏差（`.std().round(2)`）を表示し、すべての列が平均 `0.0`・標準偏差 `1.0` になっていることを確認する
2. `penguin_norm[num_cols]` の最小値（`.min().round(2)`）と最大値（`.max().round(2)`）を表示し、すべての列が最小 `0.0`・最大 `1.0` になっていることを確認する
3. 標準化後の先頭5行 `penguin_std[num_cols].head()` を表示する

In [ ]:
# 実習③：ここにコードを書いて実行する
print("標準化後の平均:
", penguin_std[num_cols].mean().round(2))
print("標準化後の標準偏差:
", penguin_std[num_cols].std().round(2))

print("正規化後の最小値:
", penguin_norm[num_cols].min().round(2))
print("正規化後の最大値:
", penguin_norm[num_cols].max().round(2))

penguin_std[num_cols].head()

---
## 4. タスク④：対数変換と機械学習での検証（`CPS1988.csv` ＆ ペンギンAI分類）

### 4-1. 右に裾が長い数値の対数変換（`np.log`）

第22講（線形回帰）でも使う米国労働者28,155人の賃金データ（`CPS1988.csv`）を読み込む。
週給 `wage` のように少数の超高額者に引っ張られて右に大きく歪んだ変数は、自然対数 **`np.log()`** で変換すると左右対称に近い釣鐘型の分布になる。

In [ ]:
url_cps = "https://raw.githubusercontent.com/ChungWookyung/kg-jupyterlite-data-analysis/main/content/DATA/CPS1988.csv"
cps = pd.read_csv(open_url(url_cps))

# 週給 wage を自然対数 np.log() で変換する
cps["log_wage"] = np.log(cps["wage"])

print("変換前の歪度 (Skewness):", cps["wage"].skew().round(2))
print("対数変換後の歪度 (Skewness):", cps["log_wage"].skew().round(2))

In [ ]:
# 変換前の wage と対数変換後の log_wage のヒストグラムを比較する
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(cps["wage"], bins=50, color="orange", edgecolor="white")
axes[0].set_title("変換前: wage の分布")
axes[1].hist(cps["log_wage"], bins=50, color="seagreen", edgecolor="white")
axes[1].set_title("対数変換後: np.log(wage) の分布")
plt.tight_layout()
plt.show()

### 4-2. `CPS1988.csv` のカテゴリ変数も一括変換する

2種類の文字（`ethnicity`, `smsa`, `parttime`）は `.replace()` で `1 / 0` に変換し、4種類の地域（`region`）は `pd.get_dummies(..., dtype=int)` でダミー変数に展開する。

In [ ]:
cps["afam"] = cps["ethnicity"].replace({"afam": 1, "cauc": 0})
cps["smsa"] = cps["smsa"].replace({"yes": 1, "no": 0})
cps["parttime"] = cps["parttime"].replace({"yes": 1, "no": 0})

cps_clean = pd.get_dummies(cps.drop("ethnicity", axis=1), columns=["region"], dtype=int)
cps_clean.head(3)

### 4-3. 総合検証：数値の標準化でニューラルネットワークの正解率はどう変わるか？

第23講で詳しく学ぶニューラルネットワーク（`MLPClassifier`）を使って、ペンギン3品種の分類を行う。
- **【比較1：標準化なし】** 数値4列を元の単位（`mm` と `g`）のまま学習させた場合
- **【比較2：標準化あり】** 数値4列を標準化（平均 `0`・標準偏差 `1`）してから学習させた場合

In [ ]:
from sklearn.neural_network import MLPClassifier

X_cols = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g",
          "sex", "island_Biscoe", "island_Dream", "island_Torgersen"]
Y_cols = ["species_Adelie", "species_Chinstrap", "species_Gentoo"]

# 【比較1】標準化なし（元の単位のまま学習）
train_raw = penguin_dum.sample(frac=0.8, random_state=1234)
test_raw = penguin_dum.drop(train_raw.index)

model_raw = MLPClassifier(hidden_layer_sizes=(16, 8), alpha=0.5, max_iter=5000, random_state=1234)
model_raw.fit(train_raw[X_cols], train_raw[Y_cols])
print("【標準化なし】訓練正解率:", round(model_raw.score(train_raw[X_cols], train_raw[Y_cols]) * 100, 1), "%")
print("【標準化なし】テスト正解率:", round(model_raw.score(test_raw[X_cols], test_raw[Y_cols]) * 100, 1), "%")

In [ ]:
# 【比較2】標準化あり（数値4列を平均0・標準偏差1に変換してから同じAIで学習）
penguin_sc = penguin_dum.copy()
penguin_sc[num_cols] = (penguin_sc[num_cols] - penguin_sc[num_cols].mean()) / penguin_sc[num_cols].std()

train_sc = penguin_sc.sample(frac=0.8, random_state=1234)
test_sc = penguin_sc.drop(train_sc.index)

model_sc = MLPClassifier(hidden_layer_sizes=(16, 8), alpha=0.5, max_iter=5000, random_state=1234)
model_sc.fit(train_sc[X_cols], train_sc[Y_cols])
print("【標準化あり】訓練正解率:", round(model_sc.score(train_sc[X_cols], train_sc[Y_cols]) * 100, 1), "%")
print("【標準化あり】テスト正解率:", round(model_sc.score(test_sc[X_cols], test_sc[Y_cols]) * 100, 1), "%")

### 実習④：対数変換と機械学習モデルの比較実験

次の3つを下のコードセルで実行しなさい。
1. `cps["wage"]` の平均値・中央値と、対数変換した `cps["log_wage"]` の平均値・中央値をそれぞれ `.round(2)` で表示する
2. 変換後の `cps_clean` の4つの地域ダミー列（`region_northeast`, `region_midwest`, `region_south`, `region_west`）の平均（各地域の割合）を `.mean().round(3)` で表示する
3. 上で学習したニューラルネットワーク `model_raw`（標準化なし）と `model_sc`（標準化あり）のテスト正解率（`%`）を並べて表示し、標準化の効果を確認する

In [ ]:
# 実習④：ここにコードを書いて実行する
print("wage 平均:", cps["wage"].mean().round(2), "中央値:", cps["wage"].median().round(2))
print("log_wage 平均:", cps["log_wage"].mean().round(2), "中央値:", cps["log_wage"].median().round(2))

region_cols = ["region_northeast", "region_midwest", "region_south", "region_west"]
print("各地域の割合:
", cps_clean[region_cols].mean().round(3))

print("標準化なし テスト正解率:", round(model_raw.score(test_raw[X_cols], test_raw[Y_cols]) * 100, 1), "%")
print("標準化あり テスト正解率:", round(model_sc.score(test_sc[X_cols], test_sc[Y_cols]) * 100, 1), "%")